# Video Stress Analysis Notebook

- detect emotions 
- analyze stress levels 

### 1. Setup & Configuration



In [7]:
import cv2
import os
import sys
from deepface import DeepFace
from collections import Counter

# Setup paths to access the backend modules
current_dir = os.getcwd()
backend_path = os.path.abspath(os.path.join(current_dir, ".."))
if backend_path not in sys.path:
    sys.path.append(backend_path)

from app.core.config import vid_stress
print(f"Project path added: {backend_path}")

# Resolve video path: it's relative to app/services/ in .env
# Using absolute path to avoid confusion
video_path_resolved = os.path.abspath(os.path.join(backend_path, "app", "services", vid_stress))
print(f"Resolved video path: {video_path_resolved}")

Project path added: /home/mubuntux/Dev/Briefs/Career-Pathfinder-AI-🚩/backend
Resolved video path: /home/mubuntux/Dev/Briefs/Career-Pathfinder-AI-🚩/backend/data/videos/raw/Stressed_asian_woman_speaking.mp4


### 2. Analysis Logic


In [8]:
def analyze_video_offline(video_path):
    """
    Analyzes a video to detect emotions and calculate stress/confidence scores.
    """
    # Check if file exists
    if not os.path.exists(video_path):
        print(f"Error: File {video_path} does not exist.")
        return None

    # Loading video
    cap = cv2.VideoCapture(video_path)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps == 0: fps = 30  # Default if undetermined
    
    frame_count = 0
    emotions_list = []

    print(f"--- Analyzing video: {video_path} ---")
    print("Processing (1 frame per second to optimize CPU)...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # CPU Optimization: Analyze only 1 frame per second
        if frame_count % fps == 0:
            try:
                # DeepFace Analysis
                results = DeepFace.analyze(
                    frame, 
                    actions=['emotion'], 
                    enforce_detection=False,
                    detector_backend='opencv'
                )
                
                # Get dominant emotion
                if isinstance(results, list):
                    dom_emotion = results[0]['dominant_emotion']
                else:
                    dom_emotion = results['dominant_emotion']
                
                emotions_list.append(dom_emotion)
                print(f"Second {frame_count//fps}: {dom_emotion}")
                
            except Exception as e:
                # Ignore frames where face is not detectable
                pass

        frame_count += 1
        
    cap.release()

    # Final calculations
    if not emotions_list:
        return {"error": "No face detected or unreadable video."}

    # Classification for scores
    stress_emotions = ["fear", "angry", "sad", "disgust"]
    confidence_emotions = ["happy", "neutral"]
    
    total_frames = len(emotions_list)
    stress_count = sum(1 for e in emotions_list if e in stress_emotions)
    confidence_count = sum(1 for e in emotions_list if e in confidence_emotions)
    
    emotion_dominant = Counter(emotions_list).most_common(1)[0][0]
    stress_score = round((stress_count / total_frames) * 100)
    confidence_score = round((confidence_count / total_frames) * 100)

    return {
        "emotion_dominant": emotion_dominant,
        "stress_score": f"{stress_score}%",
        "confidence_score": f"{confidence_score}%"
    }

### 3. Run Analysis


In [9]:
# Run Test using the resolved path
if video_path_resolved:
    result = analyze_video_offline(video_path_resolved)
    
    if result:
        print("\n" + "="*30)
        print("ANALYSIS RESULTS")
        print("="*30)
        print(f"Dominant Emotion : {result.get('emotion_dominant')}")
        print(f"Stress Score     : {result.get('stress_score')}")
        print(f"Confidence Score : {result.get('confidence_score')}")
        print("="*30)
else:
    print("No test video path found in configuration (vid_stress_env).")

--- Analyzing video: /home/mubuntux/Dev/Briefs/Career-Pathfinder-AI-🚩/backend/data/videos/raw/Stressed_asian_woman_speaking.mp4 ---
Processing (1 frame per second to optimize CPU)...


E0000 00:00:1774441402.219747   37131 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Second 0: angry
Second 1: angry
Second 2: angry
Second 3: angry
Second 4: angry
Second 5: sad
Second 6: angry
Second 7: sad
Second 8: fear
Second 9: sad
Second 10: sad
Second 11: sad
Second 12: happy
Second 13: sad
Second 14: sad
Second 15: surprise
Second 16: sad
Second 17: sad
Second 18: sad
Second 19: neutral
Second 20: angry
Second 21: angry
Second 22: sad
Second 23: sad
Second 24: sad
Second 25: sad
Second 26: sad
Second 27: neutral
Second 28: sad
Second 29: sad
Second 30: sad
Second 31: neutral
Second 32: sad
Second 33: sad
Second 34: neutral
Second 35: sad
Second 36: sad
Second 37: sad
Second 38: sad
Second 39: sad

ANALYSIS RESULTS
Dominant Emotion : sad
Stress Score     : 85%
Confidence Score : 12%
